In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import numpy as np
from maite_datasets.image_classification import MNIST

from dataeval import Embeddings
from dataeval.core import divergence_fnn
from dataeval.data import Limit, View
from dataeval.extractors import FlattenExtractor

In [3]:
# Load in the training mnist dataset and use the first 4000
train_ds = MNIST(root="./data/", image_set="train", download=True)
train_view = View(train_ds, Limit(4000))

# Create the extractor. It holds no state, so one instance serves every dataset below
extractor = FlattenExtractor()

# Extract the embeddings of the 4000 images
embeddings = np.asarray(Embeddings(train_view, extractor=extractor, batch_size=64))

In [4]:
data_a = embeddings[:2000]
data_b = embeddings[2000:]

In [5]:
div = divergence_fnn(data_a, data_b)
print(div)

{'divergence': np.float64(0.1855), 'errors': np.int64(1629)}


In [6]:
corrupted_ds = MNIST(root="./data", image_set="train", corruption="translate", download=True)
corrupted_view = View(corrupted_ds, Limit(2000))

# Reuse the extractor from above
corrupted_emb = np.asarray(Embeddings(corrupted_view, extractor=extractor, batch_size=64))

In [7]:
div = divergence_fnn(data_a, corrupted_emb)
print(div)

{'divergence': np.float64(0.9905), 'errors': np.int64(19)}


In [8]:
# TEST ASSERTION CELL ###
assert embeddings.shape == (4000, 784)
assert corrupted_emb.shape == (2000, 784)
assert div["divergence"] > 0.95